# 🕵️ V54: THE PURE FORCING MODEL (Anti-Leakage)\n\nNotebook ini adalah puncak dari integritas Data Science! Kita membuang semua ilusi (*Data Leakage*) yang disebabkan oleh **Random K-Fold** dan **Fitur Tinggi Air Masa Lalu** (`last_known_tma`).\n\nSebagai gantinya, kita memaksa algoritma **Local LightGBM** untuk menebak tinggi air secara jujur menggunakan **Hydrological Memory** (Akumulasi Hujan Jangka Panjang: 1 hari, 7 hari, 30 hari, hingga 60 hari). Model akan dilatih dengan murni **Time-Series Split** (80% awal kalender untuk latih, 20% akhir untuk validasi).\n\nBersiaplah melihat skor validasi yang *nyata* dan *identik* dengan skor Leaderboard Kaggle!

In [ ]:
!pip install -q lightgbm optuna
import os
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import lightgbm as lgb
import optuna
from sklearn.metrics import mean_squared_error

plt.style.use('seaborn-v0_8-whitegrid')

if not 'path' in locals():
    if os.path.exists('/kaggle/input'):
        dataset_folders = [f for f in os.listdir('/kaggle/input') if os.path.isdir(f'/kaggle/input/{f}')]
        path = f'/kaggle/input/{dataset_folders[0]}/' if dataset_folders else '/kaggle/input/'
    elif os.path.exists('/content'):
        from google.colab import drive
        drive.mount('/content/drive')
        path = '/content/drive/MyDrive/SSF-SSDS/'
    else:
        path = './'

train_df = pd.read_csv(path + 'train.csv', parse_dates=['datetime'])
test_df = pd.read_csv(path + 'test.csv')
lingkungan_df = pd.read_csv(path + 'data_pendukung/data_lingkungan.csv', parse_dates=['datetime'])
sample_sub = pd.read_csv(path + 'sample_submission.csv')
coords_df = pd.read_csv(path + 'data_pendukung/koordinat_pos.csv')

print("Data berhasil dimuat!")

## 1. Pemetaan Hulu-Hilir (Euclidean Cascade)

In [ ]:
print("Membangun Peta Aliran Air...")
elevations = train_df.groupby('nama_pos')['tma_mdpl'].mean().to_dict()
coords = coords_df.copy()
coords['elevation'] = coords['nama_pos'].map(elevations)

upstream_map = {pos: [] for pos in coords['nama_pos']}

for i, row_i in coords.iterrows():
    pos_i = row_i['nama_pos']
    elev_i = row_i['elevation']
    
    candidates = coords[coords['elevation'] < elev_i].copy()
    if len(candidates) > 0:
        dist = np.sqrt((candidates['latitude'] - row_i['latitude'])**2 + (candidates['longitude'] - row_i['longitude'])**2)
        nearest_idx = dist.idxmin()
        pos_hilir = coords.loc[nearest_idx, 'nama_pos']
        upstream_map[pos_hilir].append(pos_i)

## 2. Injeksi "Hydrological Memory" (Pengganti `last_known_tma`)\nKita menghitung curah hujan hingga 60 hari (1440 jam) ke belakang sebagai ingatan jangka panjang dari resapan air tanah yang menopang *base level* (tinggi dasar) sungai.

In [ ]:
print("Mengekstraksi Hydrological Memory & Fisika Tanah...")
lingkungan_df = lingkungan_df.sort_values(['nama_pos', 'datetime']).reset_index(drop=True)

def build_env_features(env_df):
    agg_df = env_df[['datetime', 'nama_pos']].copy()
    grouped = env_df.groupby('nama_pos')
    
    # 📌 THE HYDROLOGICAL MEMORY (Substitusi TMA Lag)
    agg_df['rain_sum_24h'] = grouped['rainfall_mm'].transform(lambda x: x.rolling(24, min_periods=1).sum())     # Flash Runoff
    agg_df['rain_sum_168h'] = grouped['rainfall_mm'].transform(lambda x: x.rolling(168, min_periods=1).sum())   # Weekly Runoff
    agg_df['rain_sum_720h'] = grouped['rainfall_mm'].transform(lambda x: x.rolling(720, min_periods=1).sum())   # Monthly Baseflow
    agg_df['rain_sum_1440h'] = grouped['rainfall_mm'].transform(lambda x: x.rolling(1440, min_periods=1).sum()) # 60-Day Deep Groundwater
    
    # Peluruhan Tanah
    agg_df['rain_ewm_72h'] = grouped['rainfall_mm'].transform(lambda x: x.ewm(halflife=72, ignore_na=True).mean())
    
    if 'soil_moisture_100_255cm' in env_df.columns:
        agg_df['deep_soil_ewm_336h'] = grouped['soil_moisture_100_255cm'].transform(lambda x: x.ewm(halflife=336, ignore_na=True).mean())
        agg_df['soil_0_7cm_sum_168h'] = grouped['soil_moisture_0_7cm'].transform(lambda x: x.rolling(168, min_periods=1).sum())
        
    return agg_df

env_feat = build_env_features(lingkungan_df)

## 3. Pure Forcing Dataset (Penghapusan Total Target Lag)

In [ ]:
def prepare_dataset_pure_forcing(base_df, env_feat, upstream_map):
    df = base_df.copy()
    
    df['hour'] = df['datetime'].dt.hour
    df['month'] = df['datetime'].dt.month
    df['dayofyear'] = df['datetime'].dt.dayofyear
    df['hour_sin'] = np.sin(2 * np.pi * df['hour']/24.0)
    df['hour_cos'] = np.cos(2 * np.pi * df['hour']/24.0)
    
    df = pd.merge(df, env_feat, on=['datetime', 'nama_pos'], how='left')
    
    # 🚨 PENTING: KITA TIDAK LAGI MENGINJEKSI tma_lag MAUPUN last_known_tma 🚨
    
    print("  -> Menyuntikkan Kaskade Hulu dengan Waktu Tempuh (Multi-Lag)...")
    upstream_dfs = []
    
    for pos, upstreams in upstream_map.items():
        if len(upstreams) > 0:
            up_df = df[df['nama_pos'].isin(upstreams)]
            agg = up_df.groupby('datetime')[['rain_sum_168h', 'rain_sum_72h' if 'rain_sum_72h' in df.columns else 'rain_sum_24h']].mean().reset_index()
        else:
            up_df = df[df['nama_pos'] == pos]
            agg = up_df[['datetime', 'rain_sum_168h', 'rain_sum_24h']].copy()
            
        agg = agg.rename(columns={
            'rain_sum_168h': 'upstream_rain_sum_168h',
            'rain_sum_24h': 'upstream_rain_sum_24h'
        })
        
        # 🕰️ MULTI-LAG THE TIME-MACHINE 🕰️
        agg['upstream_rain_168h_lag_1d'] = agg['upstream_rain_sum_168h'].shift(24) 
        agg['upstream_rain_168h_lag_2d'] = agg['upstream_rain_sum_168h'].shift(48)
        agg['upstream_rain_168h_lag_3d'] = agg['upstream_rain_sum_168h'].shift(72)
        
        agg['nama_pos'] = pos
        upstream_dfs.append(agg)
        
    all_upstream = pd.concat(upstream_dfs, ignore_index=True)
    df = pd.merge(df, all_upstream, on=['datetime', 'nama_pos'], how='left')
    
    # Sort secara kronologis (SANGAT PENTING UNTUK TIME-SERIES SPLIT!)
    df = df.sort_values(['nama_pos', 'datetime']).reset_index(drop=True)
    
    return df

print("Menyusun Train Set (Pure Forcing)...")
train_full = prepare_dataset_pure_forcing(train_df, env_feat, upstream_map)

print("Menyusun Test Set (Pure Forcing)...")
test_parsed = test_df.copy()
split_info = test_parsed['id'].str.split(' - ', n=1, expand=True)
test_parsed['datetime'] = pd.to_datetime(split_info[0])
test_parsed['nama_pos'] = split_info[1]
test_full = prepare_dataset_pure_forcing(test_parsed, env_feat, upstream_map)

train_full = train_full.dropna(subset=['tma_mdpl']).reset_index(drop=True)

drop_cols = ['datetime', 'nama_pos', 'tma_mdpl', 'id']
features = [c for c in train_full.columns if c not in drop_cols]
print(f"Total Fitur Murni Hidrologi: {len(features)}")


## 4. Pelatihan 30 Model Lokal (Time-Series Split Sejati) & Top-3 Ensemble

In [ ]:
stations = train_full['nama_pos'].unique()
station_models = {}
station_scores = {}

optuna.logging.set_verbosity(optuna.logging.ERROR)
N_TRIALS = 15

print(f"Memulai Optuna LightGBM (Anti-Leakage) untuk 30 Stasiun...\n")

for st in stations:
    # Data sudah diurutkan berdasarkan waktu (kronologis)
    X_st = train_full[train_full['nama_pos'] == st][features]
    y_st = train_full[train_full['nama_pos'] == st]['tma_mdpl']
    
    # 🔥 TIME-SERIES SPLIT MURNI (80% AWAL vs 20% AKHIR) 🔥
    split_idx = int(len(X_st) * 0.8)
    X_tr_base, y_tr_base = X_st.iloc[:split_idx], np.clip(y_st.iloc[:split_idx], 0.0001, None)
    X_va_base, y_va_base = X_st.iloc[split_idx:], np.clip(y_st.iloc[split_idx:], 0.0001, None)
    
    def objective(trial):
        params = {
            'objective': 'tweedie', 
            'tweedie_variance_power': trial.suggest_float('tweedie_variance_power', 1.1, 1.9),
            'metric': 'rmse', 
            'boosting_type': 'gbdt',
            'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
            'num_leaves': trial.suggest_int('num_leaves', 15, 63),
            'max_depth': trial.suggest_int('max_depth', 3, 10),
            'feature_fraction': trial.suggest_float('feature_fraction', 0.6, 1.0),
            'bagging_fraction': trial.suggest_float('bagging_fraction', 0.6, 1.0),
            'bagging_freq': 1,
            'verbose': -1,
            'random_state': 42
        }
        
        dtrain = lgb.Dataset(X_tr_base, label=y_tr_base)
        dval = lgb.Dataset(X_va_base, label=y_va_base, reference=dtrain)
        
        callbacks = [lgb.early_stopping(30, verbose=False)]
        model = lgb.train(params, dtrain, valid_sets=[dval], num_boost_round=800, callbacks=callbacks)
        
        preds = model.predict(X_va_base)
        return np.sqrt(mean_squared_error(y_va_base, preds))

    study = optuna.create_study(direction='minimize')
    study.optimize(objective, n_trials=N_TRIALS)
    
    top_trials = sorted(study.trials, key=lambda t: t.value)[:3]
    models = []
    
    for i, trial in enumerate(top_trials):
        params = trial.params.copy()
        params.update({'objective': 'tweedie', 'metric': 'rmse', 'verbose': -1, 'random_state': 42+i})
        
        dtrain = lgb.Dataset(X_tr_base, label=y_tr_base)
        dval = lgb.Dataset(X_va_base, label=y_va_base, reference=dtrain)
        
        callbacks = [lgb.early_stopping(50, verbose=False)]
        model = lgb.train(params, dtrain, valid_sets=[dval], num_boost_round=1000, callbacks=callbacks)
        models.append(model)
        
    # Hitung RMSE Ensemble
    ensemble_preds = np.zeros(len(X_va_base))
    for m in models:
        ensemble_preds += m.predict(X_va_base) / 3.0
        
    avg_rmse = np.sqrt(mean_squared_error(y_va_base, ensemble_preds))
    
    station_models[st] = models
    station_scores[st] = avg_rmse
    
    print(f"[{st}] - RMSE Waktu Nyata (Validation 20% Akhir): {avg_rmse:.4f}")

global_rmse = np.mean(list(station_scores.values()))
print(f"\n🏅 TRUE VALIDATION RMSE ACROSS 30 STATIONS: {global_rmse:.4f}")

## 5. Eksekusi Test Set & Penyerahan (The Honest Truth)

In [ ]:
# ================= BAGIAN TEST SET =================
test_preds = np.zeros(len(test_full))

for st in stations:
    idx = test_full['nama_pos'] == st
    if idx.sum() == 0: continue
        
    X_test_st = test_full.loc[idx, features]
    
    st_preds = np.zeros(len(X_test_st))
    for m in station_models[st]:
        st_preds += m.predict(X_test_st) / 3.0
        
    test_full.loc[idx, 'tma_mdpl'] = st_preds

sub_pred = test_full[['id', 'tma_mdpl']].copy()
sub = pd.merge(sample_sub[['id']], sub_pred, on='id', how='left')
sub['tma_mdpl'] = sub['tma_mdpl'].fillna(method='ffill').fillna(method='bfill')

output_file = 'submission_V54_Pure_Forcing_Honest.csv'
sub.to_csv(output_file, index=False)
print(f"\n✅ Selesai! File '{output_file}' siap disubmit. Inilah prediksi masa depan tanpa mencontek!")